In [ ]:
!pip install -q -U langchain langchain-community langchain-openai langchain-huggingface faiss-cpu pypdf sentence-transformers langchain-text-splitters

In [ ]:
# ==========================================================
# 1. DEPENDENCIES & ENVIRONMENT SETUP
# ==========================================================
import os
import warnings
from google.colab import drive

# Modern import paths for LangChain components
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

# Robust import for RetrievalQA
try:
    from langchain.chains import RetrievalQA
except ImportError:
    from langchain.chains.retrieval_qa.base import RetrievalQA

warnings.filterwarnings("ignore")

# Securely mount Google Drive
drive.mount('/content/drive', force_remount=True)

# ==========================================================
# 2. DATA INGESTION & SEMANTIC CHUNKING
# ==========================================================
DOC_PATH = '/content/drive/MyDrive/week2/data/'

if not os.path.exists(DOC_PATH):
    os.makedirs(DOC_PATH)
    print(f"⚠️ Created folder at {DOC_PATH}. Please upload your PDFs there!")

loader = DirectoryLoader(DOC_PATH, glob="*.pdf", loader_cls=PyPDFLoader)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    add_start_index=True
)
chunks = text_splitter.split_documents(documents)
print(f"✅ Status: Successfully created {len(chunks)} chunks.")

# ==========================================================
# 3. VECTOR EMBEDDINGS & FAISS INDEXING
# ==========================================================
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vector_db = FAISS.from_documents(chunks, embeddings)
DB_SAVE_PATH = "/content/drive/MyDrive/week2/data"
vector_db.save_local(DB_SAVE_PATH)
print(f"✅ Status: FAISS index persisted at {DB_SAVE_PATH}")

# ==========================================================
# 4. RAG CHAIN CONFIGURATION
# ==========================================================
system_template = """You are a professional Customer Support Agent for Everstorm Outfitters.
Answer the user's question ONLY based on the provided context.
If the information is not in the context, politely state that you do not have that information.

Context:
{context}

Question: {question}

Final Answer:"""

PROMPT = PromptTemplate(template=system_template, input_variables=["context", "question"])

API_KEY = "ENTER YOUR KEY"

llm = ChatOpenAI(
    model_name="google/gemma-2-9b-it",
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key=API_KEY,
    default_headers={
        "HTTP-Referer": "https://colab.research.google.com/",
        "X-Title": "Everstorm RAG Lab"
    }
)

rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vector_db.as_retriever(search_kwargs={"k": 3}),
    chain_type_kwargs={"prompt": PROMPT},
    return_source_documents=True
)

# ==========================================================
# 5. INFERENCE & VERIFICATION
# ==========================================================
test_query = "What are the rules for international shipping to Canada?"

try:
    response = rag_chain.invoke({"query": test_query})
    print(f"\n[AI RESPONSE]:\n{response['result']}")
    print("\n[SOURCES CITED]:")
    for doc in response['source_documents']:
        source_name = doc.metadata.get('source', 'Unknown')
        print(f"- Source: {source_name}")
except Exception as e:
    print(f"❌ Error during inference: {e}")